In [6]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

df = pd.read_csv('data/Bengaluru_House_Data.csv')
df = df[['location','size','total_sqft','bath','price']].dropna()
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(int)

def to_num(x):
    try:
        return float(x)
    except:
        try:
            a, b = x.split('-')
            return (float(a) + float(b)) / 2
        except:
            return np.nan

df['total_sqft'] = df['total_sqft'].apply(to_num)
df = df.dropna()
df = df[df['total_sqft'] / df['bhk'] >= 300]
df['price_per_sqft'] = df['price']*100000 / df['total_sqft']

def remove_pps_outliers(df):
    out = pd.DataFrame()
    for key, sub in df.groupby('location'):
        m, s = sub['price_per_sqft'].mean(), sub['price_per_sqft'].std()
        reduced = sub[(sub['price_per_sqft'] > (m-s)) & (sub['price_per_sqft'] <= (m+s))]
        out = pd.concat([out, reduced], ignore_index=True)
    return out

df = remove_pps_outliers(df)

def remove_bhk_outliers(df):
    exclude = np.array([])
    for loc, sub in df.groupby('location'):
        stats = {}
        for bhk, bhk_df in sub.groupby('bhk'):
            stats[bhk] = {'mean': bhk_df['price_per_sqft'].mean(), 'count': bhk_df.shape[0]}
        for bhk, bhk_df in sub.groupby('bhk'):
            s = stats.get(bhk-1)
            if s and s['count'] > 5:
                exclude = np.append(exclude, bhk_df[bhk_df['price_per_sqft'] < s['mean']].index.values)
    return df.drop(exclude, axis='index')

df = remove_bhk_outliers(df)
df = df[df['bath'] < df['bhk'] + 2]
df = df.drop(columns=['price_per_sqft'])

X = df[['location','total_sqft','bath','bhk']]
y = df['price']   # price is in Lakhs

pre = ColumnTransformer([('loc', OneHotEncoder(handle_unknown='ignore'), ['location'])], remainder='passthrough')
model = Pipeline([('pre', pre), ('rf', GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42))])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(Xtr, ytr)
print("R2:", r2_score(yte, model.predict(Xte)))

joblib.dump({'model': model, 'locations': sorted(top) + ['other']}, 'model_india.pkl', compress=3)

R2: 0.7926346412332433


['model_india.pkl']